<a href="https://colab.research.google.com/github/Dania-Yasir/flyrak-project/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [ ]:
%pip install -q duckdb huggingface_hub pandas numpy scikit-learn matplotlib

In [ ]:
import calendar
import json
import warnings
from pathlib import Path
from getpass import getpass

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from IPython.display import display
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier

warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_STATE = 42
N_SPLITS = 5
DECLINE_THRESHOLD_PCT = -20.0

np.random.seed(RANDOM_STATE)

OUTPUT_DIR = Path("work/outputs")
FIGURE_DIR = Path("work/figures")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print("Setup complete.")

Setup complete.


In [ ]:
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = None

if not HF_TOKEN:
    HF_TOKEN = getpass("Enter your Hugging Face READ token: ")

if not HF_TOKEN:
    raise ValueError("A Hugging Face READ token is required.")

con = duckdb.connect()

safe_token = HF_TOKEN.replace("'", "''")

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{safe_token}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

print("Connected successfully.")

Connected successfully.


In [ ]:
# Feature groups are declared once and reused for March and the temporal holdout.

ANCHOR_FEATURES = [
    "imp_momentum_pct",
    "imp_momentum_log",
]

LEVEL_FEATURES = [
    "log_imp_first_half",
    "log_clicks_first_half",
    "ctr_first_half",
    "avg_position_first_half",
    "active_rate_first_half",
]

TREND_SHAPE_FEATURES = [
    "imp_step12_log",
    "imp_step23_log",
    "imp_acceleration",
    "imp_declining_steps",
    "imp_recent_vs_prior_log",
    "click_step23_log",
    "ctr_change_5d_pp",
    "position_change_5d",
    "active_rate_change_5d",
    "has_position_shape",
]

CLIENT_CONTEXT_FEATURES = [
    "log_imp_client_percentile",
    "ctr_client_percentile",
    "position_client_percentile",
    "recent_vs_prior_client_percentile",
]

CONTEXT_FEATURES = (
    LEVEL_FEATURES
    + TREND_SHAPE_FEATURES
    + CLIENT_CONTEXT_FEATURES
)
FULL_FEATURES = ANCHOR_FEATURES + CONTEXT_FEATURES

print("Momentum anchor features:", len(ANCHOR_FEATURES))
print("Context features:", len(CONTEXT_FEATURES))
print("Full features:", len(FULL_FEATURES))

Momentum anchor features: 2
Context features: 19
Full features: 21


In [ ]:
def safe_pct_change(new_value, old_value):
    denom = np.maximum(np.asarray(old_value, dtype=float), 1.0)
    return (
        (
            np.asarray(new_value, dtype=float)
            - np.asarray(old_value, dtype=float)
        )
        / denom
    ) * 100.0


def build_month_model_frame(month_str):
    """
    Build one month's feature population and outcome-evaluable cohort.

    Feature window: days 1-15.
    Outcome window: days 16-end of month.

    Client-relative ranks are computed on the pre-outcome feature population,
    before filtering on future coverage.
    """
    year, month = map(int, month_str.split("-"))
    last_day = calendar.monthrange(year, month)[1]
    future_days = last_day - 15

    d1 = f"{month_str}-01"
    d5 = f"{month_str}-05"
    d6 = f"{month_str}-06"
    d7 = f"{month_str}-07"
    d8 = f"{month_str}-08"
    d10 = f"{month_str}-10"
    d11 = f"{month_str}-11"
    d15 = f"{month_str}-15"
    d16 = f"{month_str}-16"
    dend = f"{month_str}-{last_day:02d}"

    fact = (
        f"read_parquet("
        f"'{REL}/fact_content_daily_performance/month={month_str}/*.parquet'"
        f")"
    )

    query = f"""
    SELECT
        client_hash_id AS client_id,
        content_hash_id AS content_id,

        SUM(CASE WHEN report_date BETWEEN DATE '{d1}' AND DATE '{d5}'
                 THEN COALESCE(gsc_impressions, 0) ELSE 0 END) AS imp_w1,
        SUM(CASE WHEN report_date BETWEEN DATE '{d1}' AND DATE '{d5}'
                 THEN COALESCE(gsc_clicks, 0) ELSE 0 END) AS clicks_w1,
        SUM(CASE WHEN report_date BETWEEN DATE '{d1}' AND DATE '{d5}'
                 THEN COALESCE(gsc_sum_position, 0) ELSE 0 END) AS sum_position_w1,
        SUM(CASE WHEN report_date BETWEEN DATE '{d1}' AND DATE '{d5}'
                      AND COALESCE(gsc_impressions, 0) > 0
                 THEN 1 ELSE 0 END) AS active_days_w1,

        SUM(CASE WHEN report_date BETWEEN DATE '{d6}' AND DATE '{d10}'
                 THEN COALESCE(gsc_impressions, 0) ELSE 0 END) AS imp_w2,
        SUM(CASE WHEN report_date BETWEEN DATE '{d6}' AND DATE '{d10}'
                 THEN COALESCE(gsc_clicks, 0) ELSE 0 END) AS clicks_w2,
        SUM(CASE WHEN report_date BETWEEN DATE '{d6}' AND DATE '{d10}'
                 THEN COALESCE(gsc_sum_position, 0) ELSE 0 END) AS sum_position_w2,
        SUM(CASE WHEN report_date BETWEEN DATE '{d6}' AND DATE '{d10}'
                      AND COALESCE(gsc_impressions, 0) > 0
                 THEN 1 ELSE 0 END) AS active_days_w2,

        SUM(CASE WHEN report_date BETWEEN DATE '{d11}' AND DATE '{d15}'
                 THEN COALESCE(gsc_impressions, 0) ELSE 0 END) AS imp_w3,
        SUM(CASE WHEN report_date BETWEEN DATE '{d11}' AND DATE '{d15}'
                 THEN COALESCE(gsc_clicks, 0) ELSE 0 END) AS clicks_w3,
        SUM(CASE WHEN report_date BETWEEN DATE '{d11}' AND DATE '{d15}'
                 THEN COALESCE(gsc_sum_position, 0) ELSE 0 END) AS sum_position_w3,
        SUM(CASE WHEN report_date BETWEEN DATE '{d11}' AND DATE '{d15}'
                      AND COALESCE(gsc_impressions, 0) > 0
                 THEN 1 ELSE 0 END) AS active_days_w3,

        SUM(CASE WHEN report_date BETWEEN DATE '{d1}' AND DATE '{d7}'
                 THEN COALESCE(gsc_impressions, 0) ELSE 0 END) AS imp_early7,
        SUM(CASE WHEN report_date BETWEEN DATE '{d8}' AND DATE '{d15}'
                 THEN COALESCE(gsc_impressions, 0) ELSE 0 END) AS imp_late8,

        SUM(CASE WHEN report_date BETWEEN DATE '{d1}' AND DATE '{d15}'
                      AND COALESCE(gsc_data_available, FALSE)
                 THEN 1 ELSE 0 END) AS gsc_days_features,

        SUM(CASE WHEN report_date BETWEEN DATE '{d16}' AND DATE '{dend}'
                      AND COALESCE(gsc_data_available, FALSE)
                 THEN 1 ELSE 0 END) AS gsc_days_future,

        SUM(CASE WHEN report_date BETWEEN DATE '{d16}' AND DATE '{dend}'
                 THEN COALESCE(gsc_impressions, 0) ELSE 0 END) AS imp_future

    FROM {fact}
    WHERE report_date BETWEEN DATE '{d1}' AND DATE '{dend}'
    GROUP BY client_hash_id, content_hash_id
    """

    raw = con.execute(query).df()

    feature_pop = raw[
        raw["gsc_days_features"] == 15
    ].copy()

    feature_pop["imp_first_half"] = (
        feature_pop["imp_w1"]
        + feature_pop["imp_w2"]
        + feature_pop["imp_w3"]
    )
    feature_pop["clicks_first_half"] = (
        feature_pop["clicks_w1"]
        + feature_pop["clicks_w2"]
        + feature_pop["clicks_w3"]
    )
    feature_pop["sum_position_first_half"] = (
        feature_pop["sum_position_w1"]
        + feature_pop["sum_position_w2"]
        + feature_pop["sum_position_w3"]
    )
    feature_pop["active_days_first_half"] = (
        feature_pop["active_days_w1"]
        + feature_pop["active_days_w2"]
        + feature_pop["active_days_w3"]
    )

    feature_pop = feature_pop[
        feature_pop["imp_first_half"] > 0
    ].copy()

    feature_pop["avg_position_first_half"] = (
        feature_pop["sum_position_first_half"]
        / feature_pop["imp_first_half"]
    )
    feature_pop = feature_pop[
        feature_pop["avg_position_first_half"].notna()
        & (feature_pop["avg_position_first_half"] > 0)
    ].copy()

    # Stable level features.
    feature_pop["log_imp_first_half"] = np.log1p(
        feature_pop["imp_first_half"]
    )
    feature_pop["log_clicks_first_half"] = np.log1p(
        feature_pop["clicks_first_half"]
    )
    feature_pop["ctr_first_half"] = (
        feature_pop["clicks_first_half"]
        / feature_pop["imp_first_half"]
    ) * 100.0
    feature_pop["active_rate_first_half"] = (
        feature_pop["active_days_first_half"] / 15.0
    )

    # Momentum anchor.
    feature_pop["avg_daily_imp_early7"] = (
        feature_pop["imp_early7"] / 7.0
    )
    feature_pop["avg_daily_imp_late8"] = (
        feature_pop["imp_late8"] / 8.0
    )
    feature_pop["imp_momentum_pct"] = safe_pct_change(
        feature_pop["avg_daily_imp_late8"],
        feature_pop["avg_daily_imp_early7"],
    )
    feature_pop["imp_momentum_log"] = (
        np.log1p(feature_pop["avg_daily_imp_late8"])
        - np.log1p(feature_pop["avg_daily_imp_early7"])
    )

    # Three equal 5-day blocks.
    for w in (1, 2, 3):
        feature_pop[f"avg_daily_imp_w{w}"] = (
            feature_pop[f"imp_w{w}"] / 5.0
        )
        feature_pop[f"avg_daily_clicks_w{w}"] = (
            feature_pop[f"clicks_w{w}"] / 5.0
        )
        feature_pop[f"ctr_w{w}"] = np.where(
            feature_pop[f"imp_w{w}"] > 0,
            (
                feature_pop[f"clicks_w{w}"]
                / feature_pop[f"imp_w{w}"]
            ) * 100.0,
            0.0,
        )
        feature_pop[f"avg_position_w{w}"] = np.where(
            feature_pop[f"imp_w{w}"] > 0,
            (
                feature_pop[f"sum_position_w{w}"]
                / feature_pop[f"imp_w{w}"]
            ),
            np.nan,
        )
        feature_pop[f"active_rate_w{w}"] = (
            feature_pop[f"active_days_w{w}"] / 5.0
        )

    feature_pop["imp_step12_log"] = (
        np.log1p(feature_pop["avg_daily_imp_w2"])
        - np.log1p(feature_pop["avg_daily_imp_w1"])
    )
    feature_pop["imp_step23_log"] = (
        np.log1p(feature_pop["avg_daily_imp_w3"])
        - np.log1p(feature_pop["avg_daily_imp_w2"])
    )
    feature_pop["imp_acceleration"] = (
        feature_pop["imp_step23_log"]
        - feature_pop["imp_step12_log"]
    )
    feature_pop["imp_declining_steps"] = (
        (
            feature_pop["avg_daily_imp_w2"]
            < feature_pop["avg_daily_imp_w1"]
        ).astype(int)
        + (
            feature_pop["avg_daily_imp_w3"]
            < feature_pop["avg_daily_imp_w2"]
        ).astype(int)
    )
    feature_pop["imp_recent_vs_prior_log"] = (
        np.log1p(feature_pop["avg_daily_imp_w3"])
        - np.log1p(
            (
                feature_pop["avg_daily_imp_w1"]
                + feature_pop["avg_daily_imp_w2"]
            ) / 2.0
        )
    )
    feature_pop["click_step23_log"] = (
        np.log1p(feature_pop["avg_daily_clicks_w3"])
        - np.log1p(feature_pop["avg_daily_clicks_w2"])
    )
    feature_pop["ctr_change_5d_pp"] = (
        feature_pop["ctr_w3"]
        - feature_pop["ctr_w1"]
    )
    feature_pop["position_change_5d"] = (
        feature_pop["avg_position_w3"]
        - feature_pop["avg_position_w1"]
    ).fillna(0.0)
    feature_pop["has_position_shape"] = (
        feature_pop["avg_position_w1"].notna()
        & feature_pop["avg_position_w3"].notna()
    ).astype(int)
    feature_pop["active_rate_change_5d"] = (
        feature_pop["active_rate_w3"]
        - feature_pop["active_rate_w1"]
    )

    # Client-relative context uses PRE-OUTCOME data only.
    groups = feature_pop.groupby("client_id")
    feature_pop["log_imp_client_percentile"] = (
        groups["log_imp_first_half"].rank(pct=True)
    )
    feature_pop["ctr_client_percentile"] = (
        groups["ctr_first_half"].rank(pct=True)
    )
    feature_pop["position_client_percentile"] = (
        groups["avg_position_first_half"].rank(pct=True)
    )
    feature_pop["recent_vs_prior_client_percentile"] = (
        groups["imp_recent_vs_prior_log"].rank(pct=True)
    )

    model = feature_pop[
        feature_pop["gsc_days_future"] == future_days
    ].copy().reset_index(drop=True)

    model["avg_daily_imp_first_half"] = (
        model["imp_first_half"] / 15.0
    )
    model["avg_daily_imp_future"] = (
        model["imp_future"] / float(future_days)
    )
    model["impression_change_pct"] = (
        (
            model["avg_daily_imp_future"]
            - model["avg_daily_imp_first_half"]
        )
        / model["avg_daily_imp_first_half"]
    ) * 100.0
    model["is_declining_proxy"] = (
        model["impression_change_pct"]
        < DECLINE_THRESHOLD_PCT
    ).astype(int)

    info = {
        "month": month_str,
        "last_day": last_day,
        "future_days": future_days,
        "raw_rows": len(raw),
        "feature_rows": len(feature_pop),
        "model_rows": len(model),
        "clients": model["client_id"].nunique(),
    }

    return raw, feature_pop, model, info

In [ ]:
def make_balanced_client_splits(
    frame,
    n_splits=N_SPLITS,
    random_state=RANDOM_STATE,
    n_restarts=500,
):
    """
    Client-disjoint K-fold splitter with an explicit client-count constraint.

    Guarantees:
    - each client appears in exactly one test fold;
    - fold client counts differ by at most one.

    Optimization target:
    - balance total rows;
    - balance positive rows;
    - balance decline base rate.

    Model performance is NEVER used when choosing the assignment.
    """
    stats = (
        frame.groupby("client_id")[target_col]
        .agg(rows="size", positives="sum")
        .reset_index()
    )
    stats["base_rate"] = (
        stats["positives"] / stats["rows"]
    )

    n_clients = len(stats)
    n_splits = min(int(n_splits), int(n_clients))

    if n_splits < 2:
        raise ValueError("At least two clients are required.")

    # Exact fold capacities.
    # Example: 34 clients, 5 folds -> [7, 7, 7, 7, 6].
    base = n_clients // n_splits
    remainder = n_clients % n_splits

    default_capacities = np.array(
        [
            base + (1 if i < remainder else 0)
            for i in range(n_splits)
        ],
        dtype=int,
    )

    total_rows = float(stats["rows"].sum())
    total_pos = float(stats["positives"].sum())
    global_rate = total_pos / total_rows

    target_rows = total_rows / n_splits
    target_pos = total_pos / n_splits

    rng_master = np.random.default_rng(random_state)
    best = None

    def final_cost(fold_rows, fold_pos):
        fold_rows = np.asarray(fold_rows, dtype=float)
        fold_pos = np.asarray(fold_pos, dtype=float)

        fold_rates = np.divide(
            fold_pos,
            fold_rows,
            out=np.zeros_like(fold_pos),
            where=fold_rows > 0,
        )

        row_imbalance = np.mean(
            ((fold_rows - target_rows) / max(target_rows, 1.0)) ** 2
        )
        pos_imbalance = np.mean(
            ((fold_pos - target_pos) / max(target_pos, 1.0)) ** 2
        )
        rate_imbalance = np.mean(
            (fold_rates - global_rate) ** 2
        )

        # Rows and positives matter most; rate balance is an additional guard.
        return (
            1.0 * row_imbalance
            + 1.25 * pos_imbalance
            + 3.0 * rate_imbalance
        )

    for restart in range(int(n_restarts)):
        rng = np.random.default_rng(
            int(rng_master.integers(0, 2**31 - 1))
        )

        # Randomly decide which folds receive the extra client(s),
        # so no fold number gets privileged.
        capacities = np.full(
            n_splits,
            base,
            dtype=int,
        )
        extra_order = rng.permutation(n_splits)
        capacities[extra_order[:remainder]] += 1

        work = stats.copy()

        # Place difficult-to-balance clients early:
        # large clients + clients with unusual base rate.
        work["_priority"] = (
            np.log1p(work["rows"])
            + 3.0 * np.abs(
                work["base_rate"] - global_rate
            )
            + rng.normal(
                0.0,
                0.05,
                size=len(work),
            )
        )
        work = (
            work.sort_values(
                "_priority",
                ascending=False,
            )
            .reset_index(drop=True)
        )

        fold_clients = [
            [] for _ in range(n_splits)
        ]
        fold_rows = np.zeros(
            n_splits,
            dtype=float,
        )
        fold_pos = np.zeros(
            n_splits,
            dtype=float,
        )

        for row in work.itertuples(index=False):
            eligible = [
                fold
                for fold in range(n_splits)
                if len(fold_clients[fold]) < capacities[fold]
            ]

            candidate_choices = []

            for fold in eligible:
                test_rows = fold_rows.copy()
                test_pos = fold_pos.copy()

                test_rows[fold] += float(row.rows)
                test_pos[fold] += float(row.positives)

                # Partial-state cost. Empty folds are intentionally penalized
                # through the row/positive imbalance terms.
                cost = final_cost(
                    test_rows,
                    test_pos,
                )

                candidate_choices.append(
                    (
                        cost,
                        rng.random(),
                        fold,
                    )
                )

            _, _, chosen_fold = min(
                candidate_choices
            )

            fold_clients[chosen_fold].append(
                row.client_id
            )
            fold_rows[chosen_fold] += float(
                row.rows
            )
            fold_pos[chosen_fold] += float(
                row.positives
            )

        score = final_cost(
            fold_rows,
            fold_pos,
        )

        if (
            best is None
            or score < best["score"]
        ):
            best = {
                "score": score,
                "fold_clients": [
                    list(x)
                    for x in fold_clients
                ],
                "fold_rows": fold_rows.copy(),
                "fold_pos": fold_pos.copy(),
            }

    splits = []

    for clients_in_fold in best["fold_clients"]:
        test_mask = frame["client_id"].isin(
            set(clients_in_fold)
        ).to_numpy()

        test_idx = np.flatnonzero(
            test_mask
        )
        train_idx = np.flatnonzero(
            ~test_mask
        )

        splits.append(
            (train_idx, test_idx)
        )

    # Verification.
    client_counts = [
        frame.iloc[test_idx]["client_id"].nunique()
        for _, test_idx in splits
    ]

    assert max(client_counts) - min(client_counts) <= 1

    all_test_clients = []
    for _, test_idx in splits:
        all_test_clients.extend(
            frame.iloc[test_idx][
                "client_id"
            ].drop_duplicates().tolist()
        )

    assert (
        len(all_test_clients)
        == len(set(all_test_clients))
        == frame["client_id"].nunique()
    )

    return splits, "BalancedClientKFold"

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42

In [ ]:
logistic_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        C=0.3,
        class_weight="balanced",
        max_iter=2000,
        random_state=RANDOM_STATE,
    )),
])

random_forest_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_leaf=25,
    max_features="sqrt",
    class_weight="balanced_subsample",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

print("Models ready.")

Models ready.


In [ ]:
raw_df, feature_population, model_df, march_info = (
    build_month_model_frame("2026-03")
)

target_col = "is_declining_proxy"

print("March data:")
print(march_info)
print("Decline rate:", f"{model_df[target_col].mean():.1%}")
print("Model rows:", f"{len(model_df):,}")
print("Clients:", model_df["client_id"].nunique())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March data:
{'month': '2026-03', 'last_day': 31, 'future_days': 16, 'raw_rows': 331437, 'feature_rows': 68288, 'model_rows': 61795, 'clients': 34}
Decline rate: 32.4%
Model rows: 61,795
Clients: 34


#OOF action queue

In [ ]:
grouped_splits, grouped_strategy = make_balanced_client_splits(
    model_df,
    n_splits=N_SPLITS,
    random_state=RANDOM_STATE,
    n_restarts=500,
)

queue_frames = []

QUEUE_SIGNAL_COLS = [
    "client_id",
    "content_id",
    target_col,
    "imp_first_half",
    "clicks_first_half",
    "log_imp_first_half",
    "ctr_first_half",
    "avg_position_first_half",
    "active_rate_first_half",
    "imp_momentum_pct",
    "imp_momentum_log",
    "imp_step12_log",
    "imp_step23_log",
    "imp_acceleration",
    "imp_declining_steps",
    "imp_recent_vs_prior_log",
    "click_step23_log",
    "ctr_change_5d_pp",
    "position_change_5d",
    "active_rate_change_5d",
    "log_imp_client_percentile",
    "ctr_client_percentile",
    "position_client_percentile",
    "recent_vs_prior_client_percentile",
]

for fold, (train_idx, test_idx) in enumerate(grouped_splits, start=1):

    train_part = model_df.iloc[train_idx].copy()
    test_part = model_df.iloc[test_idx].copy()

    rf = clone(random_forest_model)

    rf.fit(
        train_part[FULL_FEATURES],
        train_part[target_col],
    )

    rf_scores = rf.predict_proba(
        test_part[FULL_FEATURES]
    )[:, 1]

    fold_queue = test_part[QUEUE_SIGNAL_COLS].copy()

    fold_queue["fold"] = fold
    fold_queue["model_risk_score"] = rf_scores

    # Make scores comparable across folds without assuming
    # identical probability calibration.
    fold_queue["risk_percentile"] = (
        pd.Series(rf_scores, index=fold_queue.index)
        .rank(method="average", pct=True)
    )

    queue_frames.append(fold_queue)

action_queue = pd.concat(queue_frames, ignore_index=True)

assert len(action_queue) == len(model_df)
assert action_queue["content_id"].duplicated().sum() == 0

print("OOF rows:", f"{len(action_queue):,}")
print("Duplicate content IDs:",
      action_queue["content_id"].duplicated().sum())

display(
    action_queue[
        [
            "fold",
            "model_risk_score",
            "risk_percentile",
            "imp_momentum_pct",
            "imp_declining_steps",
            "avg_position_first_half",
            target_col,
        ]
    ]
    .sort_values("risk_percentile", ascending=False)
    .head(10)
)

OOF rows: 61,795
Duplicate content IDs: 0


,fold,model_risk_score,risk_percentile,imp_momentum_pct,imp_declining_steps,avg_position_first_half,is_declining_proxy
58980,5,0.928680,1.000000,-91.320375,2,6.439024,1
42568,4,0.907982,1.000000,-65.891873,2,6.557978,1
25410,3,0.916179,1.000000,-82.500000,1,2.688264,0
7580,1,0.931780,1.000000,-66.123188,2,13.919403,1
18502,2,0.909352,1.000000,-75.921763,1,8.971791,1
39218,4,0.907871,0.999940,-60.098974,2,7.382679,1
56516,5,0.928109,0.999922,-78.888889,2,6.595908,1
18059,2,0.903424,0.999909,-84.264305,2,9.069284,1
6920,1,0.926008,0.999907,-62.814703,1,9.990813,1
22920,3,0.905804,0.999905,-54.434404,2,0.188380,1


#Reason codes

In [ ]:
def build_reason_codes(row):
    reasons = []

    # Absolute recent deterioration
    if row["imp_momentum_pct"] <= -20:
        reasons.append("RECENT_IMPRESSION_DROP")

    # More persistent shape, not only one noisy step
    if row["imp_declining_steps"] >= 2:
        reasons.append("SUSTAINED_DECLINE_SHAPE")

    # Unusually weak movement relative to pages of the same client
    if row["recent_vs_prior_client_percentile"] <= 0.20:
        reasons.append("CLIENT_RELATIVE_DROPOFF")

    # Search position number increased = ranking position worsened
    if row["position_change_5d"] >= 2:
        reasons.append("POSITION_WORSENING")

    # CTR is weak relative to other pages from the same client
    if row["ctr_client_percentile"] <= 0.25:
        reasons.append("LOW_RELATIVE_CTR")

    # Existing search visibility
    if (
        row["avg_position_first_half"] > 0
        and row["avg_position_first_half"] <= 20
    ):
        reasons.append("TOP20_VISIBILITY")

    # Larger visible opportunity inside the client
    if row["log_imp_client_percentile"] >= 0.75:
        reasons.append("HIGH_RELATIVE_EXPOSURE")

    if not reasons:
        reasons.append("MODEL_PATTERN_ONLY")

    return "|".join(reasons)


action_queue["reason_codes"] = action_queue.apply(
    build_reason_codes,
    axis=1,
)

action_queue[
    ["risk_percentile", "reason_codes"]
].head()

,risk_percentile,reason_codes
0,0.152878,TOP20_VISIBILITY
1,0.006663,POSITION_WORSENING|LOW_RELATIVE_CTR
2,0.332130,POSITION_WORSENING|LOW_RELATIVE_CTR
3,0.560244,LOW_RELATIVE_CTR
4,0.001110,LOW_RELATIVE_CTR


#Archetype → Action mapping

In [ ]:
def assign_archetype(row):

    reasons = set(row["reason_codes"].split("|"))
    risk = row["risk_percentile"]

    # Model risk is the first gate.
    # Only the top 20% enters the active review queue.
    if risk < 0.50:
        return "MONITOR"

    if risk < 0.80:
        return "WATCHLIST"

    # High-risk pages: reason codes determine review type.
    if {
        "RECENT_IMPRESSION_DROP",
        "SUSTAINED_DECLINE_SHAPE",
    }.issubset(reasons):
        return "SUSTAINED_DECAY"

    if (
        "POSITION_WORSENING" in reasons
        and "RECENT_IMPRESSION_DROP" in reasons
    ):
        return "RANKING_SLIPPAGE"

    if (
        "TOP20_VISIBILITY" in reasons
        and "LOW_RELATIVE_CTR" in reasons
    ):
        return "VISIBLE_LOW_CTR"

    if "CLIENT_RELATIVE_DROPOFF" in reasons:
        return "CLIENT_RELATIVE_ANOMALY"

    # High model risk without a clean heuristic explanation.
    return "HIGH_RISK_AMBIGUOUS"

In [ ]:
ACTION_MAP = {
    "SUSTAINED_DECAY":
        "CONTENT_REFRESH_REVIEW",

    "RANKING_SLIPPAGE":
        "SERP_AND_INTENT_REVIEW",

    "VISIBLE_LOW_CTR":
        "TITLE_META_CTR_REVIEW",

    "CLIENT_RELATIVE_ANOMALY":
        "MANUAL_DIAGNOSTIC_REVIEW",

    "HIGH_RISK_AMBIGUOUS":
        "HUMAN_REVIEW_BEFORE_EDIT",

    "WATCHLIST":
        "WATCHLIST_NO_IMMEDIATE_EDIT",

    "MONITOR":
        "MONITOR_NO_IMMEDIATE_EDIT",
}


action_queue["archetype"] = action_queue.apply(
    assign_archetype,
    axis=1,
)

action_queue["recommended_action"] = (
    action_queue["archetype"].map(ACTION_MAP)
)
EFFORT_MAP = {
    "CONTENT_REFRESH_REVIEW": "HIGH",
    "SERP_AND_INTENT_REVIEW": "MEDIUM",
    "TITLE_META_CTR_REVIEW": "LOW",
    "MANUAL_DIAGNOSTIC_REVIEW": "MEDIUM",
    "HUMAN_REVIEW_BEFORE_EDIT": "MEDIUM",
    "WATCHLIST_NO_IMMEDIATE_EDIT": "LOW",
    "MONITOR_NO_IMMEDIATE_EDIT": "LOW",
}

#Cost/value thinking

In [ ]:
def value_tier(row):
    p = row["log_imp_client_percentile"]

    if p >= 0.75:
        return "HIGH"
    elif p >= 0.25:
        return "MEDIUM"
    return "LOW"


EFFORT_MAP = {
    "CONTENT_REFRESH_REVIEW": "HIGH",
    "SERP_AND_INTENT_REVIEW": "MEDIUM",
    "TITLE_META_CTR_REVIEW": "LOW",
    "MANUAL_DIAGNOSTIC_REVIEW": "MEDIUM",
    "HUMAN_REVIEW_BEFORE_EDIT": "MEDIUM",
    "MONITOR_NO_IMMEDIATE_EDIT": "LOW",
}


action_queue["value_tier"] = action_queue.apply(
    value_tier,
    axis=1,
)

action_queue["effort_tier"] = (
    action_queue["recommended_action"].map(EFFORT_MAP)
)

In [ ]:
display(
    action_queue[
        [
            "risk_percentile",
            "log_imp_client_percentile",
            "value_tier",
            "recommended_action",
            "effort_tier",
        ]
    ].head(10)
)

,risk_percentile,log_imp_client_percentile,value_tier,recommended_action,effort_tier
0,1.000000,0.693631,MEDIUM,SERP_AND_INTENT_REVIEW,MEDIUM
1,1.000000,0.637254,MEDIUM,CONTENT_REFRESH_REVIEW,HIGH
2,1.000000,0.573068,MEDIUM,CONTENT_REFRESH_REVIEW,HIGH
3,1.000000,0.527907,MEDIUM,SERP_AND_INTENT_REVIEW,MEDIUM
4,1.000000,0.653874,MEDIUM,CONTENT_REFRESH_REVIEW,HIGH
5,0.999940,0.633683,MEDIUM,CONTENT_REFRESH_REVIEW,HIGH
6,0.999922,0.560589,MEDIUM,CONTENT_REFRESH_REVIEW,HIGH
7,0.999909,0.374419,MEDIUM,CONTENT_REFRESH_REVIEW,HIGH
8,0.999907,0.872556,HIGH,SERP_AND_INTENT_REVIEW,MEDIUM
9,0.999905,0.476558,MEDIUM,CONTENT_REFRESH_REVIEW,HIGH


In [ ]:
VALUE_ORDER = {
    "HIGH": 3,
    "MEDIUM": 2,
    "LOW": 1,
}

action_queue["value_order"] = (
    action_queue["value_tier"].map(VALUE_ORDER)
)

action_queue = (
    action_queue
    .sort_values(
        ["risk_percentile", "value_order", "imp_first_half"],
        ascending=[False, False, False],
    )
    .reset_index(drop=True)
)

action_queue["queue_rank"] = np.arange(len(action_queue)) + 1

action_queue["priority_score"] = (
    action_queue["risk_percentile"] * 100
).round(3)

display(
    action_queue[
        [
            "queue_rank",
            "priority_score",
            "value_tier",
            "effort_tier",
            "archetype",
            "recommended_action",
            "reason_codes",
        ]
    ].head(20)
)

,queue_rank,priority_score,value_tier,effort_tier,archetype,recommended_action,reason_codes
0,1,100.000,MEDIUM,MEDIUM,RANKING_SLIPPAGE,SERP_AND_INTENT_REVIEW,RECENT_IMPRESSION_DROP|CLIENT_RELATIVE_DROPOFF...
1,2,100.000,MEDIUM,HIGH,SUSTAINED_DECAY,CONTENT_REFRESH_REVIEW,RECENT_IMPRESSION_DROP|SUSTAINED_DECLINE_SHAPE...
2,3,100.000,MEDIUM,HIGH,SUSTAINED_DECAY,CONTENT_REFRESH_REVIEW,RECENT_IMPRESSION_DROP|SUSTAINED_DECLINE_SHAPE...
3,4,100.000,MEDIUM,MEDIUM,RANKING_SLIPPAGE,SERP_AND_INTENT_REVIEW,RECENT_IMPRESSION_DROP|CLIENT_RELATIVE_DROPOFF...
4,5,100.000,MEDIUM,HIGH,SUSTAINED_DECAY,CONTENT_REFRESH_REVIEW,RECENT_IMPRESSION_DROP|SUSTAINED_DECLINE_SHAPE...
5,6,99.994,MEDIUM,HIGH,SUSTAINED_DECAY,CONTENT_REFRESH_REVIEW,RECENT_IMPRESSION_DROP|SUSTAINED_DECLINE_SHAPE...
6,7,99.992,MEDIUM,HIGH,SUSTAINED_DECAY,CONTENT_REFRESH_REVIEW,RECENT_IMPRESSION_DROP|SUSTAINED_DECLINE_SHAPE...
7,8,99.991,MEDIUM,HIGH,SUSTAINED_DECAY,CONTENT_REFRESH_REVIEW,RECENT_IMPRESSION_DROP|SUSTAINED_DECLINE_SHAPE...
8,9,99.991,HIGH,MEDIUM,RANKING_SLIPPAGE,SERP_AND_INTENT_REVIEW,RECENT_IMPRESSION_DROP|CLIENT_RELATIVE_DROPOFF...
9,10,99.990,MEDIUM,HIGH,SUSTAINED_DECAY,CONTENT_REFRESH_REVIEW,RECENT_IMPRESSION_DROP|SUSTAINED_DECLINE_SHAPE...


In [ ]:
print("Recommended action distribution:")

display(
    action_queue["recommended_action"]
    .value_counts()
    .rename_axis("recommended_action")
    .reset_index(name="pages")
)

print("\nArchetype distribution:")

display(
    action_queue["archetype"]
    .value_counts()
    .rename_axis("archetype")
    .reset_index(name="pages")
)

Recommended action distribution:


,recommended_action,pages
0,MONITOR_NO_IMMEDIATE_EDIT,30893
1,WATCHLIST_NO_IMMEDIATE_EDIT,18540
2,CONTENT_REFRESH_REVIEW,6516
3,MANUAL_DIAGNOSTIC_REVIEW,2314
4,HUMAN_REVIEW_BEFORE_EDIT,1888
5,TITLE_META_CTR_REVIEW,935
6,SERP_AND_INTENT_REVIEW,709



Archetype distribution:


,archetype,pages
0,MONITOR,30893
1,WATCHLIST,18540
2,SUSTAINED_DECAY,6516
3,CLIENT_RELATIVE_ANOMALY,2314
4,HIGH_RISK_AMBIGUOUS,1888
5,VISIBLE_LOW_CTR,935
6,RANKING_SLIPPAGE,709


In [ ]:
reason_counts = (
    action_queue["reason_codes"]
    .str.split("|")
    .explode()
    .value_counts()
    .rename_axis("reason_code")
    .reset_index(name="pages")
)

print("Reason code distribution:")

display(reason_counts)

Reason code distribution:


,reason_code,pages
0,TOP20_VISIBILITY,49035
1,LOW_RELATIVE_CTR,21615
2,RECENT_IMPRESSION_DROP,21196
3,SUSTAINED_DECLINE_SHAPE,18563
4,HIGH_RELATIVE_EXPOSURE,17157
5,POSITION_WORSENING,17125
6,CLIENT_RELATIVE_DROPOFF,12027
7,MODEL_PATTERN_ONLY,1183


In [ ]:
action_risk_audit = (
    action_queue
    .groupby(
        ["archetype", "recommended_action"],
        observed=True
    )["risk_percentile"]
    .agg(
        pages="size",
        median_risk="median",
        min_risk="min",
        max_risk="max",
    )
    .reset_index()
    .sort_values("median_risk", ascending=False)
)

display(
    action_risk_audit.style.format({
        "median_risk": "{:.1%}",
        "min_risk": "{:.1%}",
        "max_risk": "{:.1%}",
    })
)

high_risk_monitor = action_queue[
    (action_queue["risk_percentile"] >= 0.80)
    & (action_queue["archetype"] == "MONITOR")
]

print(
    "High-risk pages mapped to MONITOR:",
    len(high_risk_monitor)
)

,archetype,recommended_action,pages,median_risk,min_risk,max_risk
3,RANKING_SLIPPAGE,SERP_AND_INTENT_REVIEW,709,91.8%,80.1%,100.0%
4,SUSTAINED_DECAY,CONTENT_REFRESH_REVIEW,6516,91.6%,80.0%,100.0%
5,VISIBLE_LOW_CTR,TITLE_META_CTR_REVIEW,935,88.6%,80.0%,99.8%
0,CLIENT_RELATIVE_ANOMALY,MANUAL_DIAGNOSTIC_REVIEW,2314,87.8%,80.0%,100.0%
1,HIGH_RISK_AMBIGUOUS,HUMAN_REVIEW_BEFORE_EDIT,1888,85.8%,80.0%,99.8%
6,WATCHLIST,WATCHLIST_NO_IMMEDIATE_EDIT,18540,65.0%,50.0%,80.0%
2,MONITOR,MONITOR_NO_IMMEDIATE_EDIT,30893,25.0%,0.0%,50.0%


High-risk pages mapped to MONITOR: 0


### How to read this queue

This playbook starts from out-of-fold scores produced by the same Random Forest — Full Signal model used in the final validation.

The model score determines review priority. The reason codes describe observable pre-outcome signals that help a human understand why a page may deserve attention. They are heuristic explanations, not causal explanations and not proof that a specific content edit will improve performance.

The archetype rules are applied in order, so each page receives one primary action even when several reason codes are present. The resulting action is a review recommendation, not an automatic publishing decision.

Relative exposure is used only as a value proxy and tie-breaker. It is not a financial value estimate or validated ROI measure.

### Decay / refresh insight

The observed data supports using recent deterioration as a signal for prioritizing review. Persistent impression decline, worsening search position, and unusually weak movement relative to the client's other pages can identify cases worth examining first.

However, decline risk should not be translated directly into an automatic refresh. Earlier analysis also showed that short-term deterioration can recover and that some future declines begin without a strong warning signal.

For this playbook, a refresh recommendation therefore means **review the page for possible refresh**, not **rewrite the page automatically**. A human should first check search intent, SERP changes, CTR, business importance, content accuracy, and whether the page is actually stale.

The 80th-percentile threshold is an operational review-capacity rule, not a learned or causally validated cutoff. It limits active recommendations to the highest-risk portion of the queue, while lower-risk pages remain on the watchlist or monitoring tier.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.